<a href="https://colab.research.google.com/github/Edw12/omics_data_analysis/blob/main/Colab_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from google.colab import userdata

# Remove the directory if it already exists to ensure a clean clone
if os.path.exists('omics_data_analysis'):
    !rm -rf omics_data_analysis

# Retrieve GitHub Token from Colab Secrets
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')

# Clone the private repository using the PAT
!git clone https://{GITHUB_TOKEN}@github.com/Edw12/omics_data_analysis.git

Cloning into 'omics_data_analysis'...
remote: Enumerating objects: 58, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 58 (delta 20), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (58/58), 493.29 KiB | 3.36 MiB/s, done.
Resolving deltas: 100% (20/20), done.


In [2]:
import sys

# Add the cloned repository to sys.path so Python can find its modules
sys.path.insert(0, '/content/omics_data_analysis')

In [9]:
import os
import sys
import importlib
import pandas as pd # Required for pd.ExcelFile in the modification logic

# --- Modify Random_Forest_Analysis.py for correct imports ---
rf_analysis_path = '/content/omics_data_analysis/Functions/Random_Forest_Analysis.py'
with open(rf_analysis_path, 'r') as f:
    rf_lines = f.readlines()

new_rf_lines = []
for line in rf_lines:
    if "from load_human_metabolites import load_human_metabolites" in line:
        new_rf_lines.append("from ..sup_func.load_human_metabolites import load_human_metabolites\n")
    elif "from Plot_Model_Results import Plot_Model_Results" in line:
        new_rf_lines.append("from ..sup_func.Plot_Model_Results import Plot_Model_Results\n")
    else:
        new_rf_lines.append(line)

with open(rf_analysis_path, 'w') as f:
    f.writelines(new_rf_lines)

print(f"Modified {rf_analysis_path} for correct imports.")

# --- Modify load_human_metabolites.py for correct file path and ensure imports ---
lm_path = '/content/omics_data_analysis/sup_func/load_human_metabolites.py'
with open(lm_path, 'r') as f:
    original_lm_lines = f.readlines()

desired_imports = ['import os\n', 'import pandas as pd\n']

# Filter out existing desired imports to avoid duplication, but keep other lines
filtered_lm_lines = []
for line in original_lm_lines:
    # Check if the line (stripped) is exactly one of the desired imports
    if line.strip() not in [imp.strip() for imp in desired_imports]:
        filtered_lm_lines.append(line)

new_lm_content = []
# Add desired imports at the very beginning of the file
new_lm_content.extend(desired_imports)

# Add the filtered original content, applying the path modification
for line in filtered_lm_lines:
    if 'file = pd.ExcelFile("human metabolites transposed.xlsx")' in line:
        # Construct the full path using os.path.join for robustness
        new_lm_content.append("    file_path = os.path.join(os.path.dirname(os.path.dirname(__file__)), 'Data', 'human metabolites transposed.xlsx')\n")
        new_lm_content.append("    file = pd.ExcelFile(file_path)\n")
    else:
        new_lm_content.append(line)

with open(lm_path, 'w') as f:
    f.writelines(new_lm_content)

print(f"Modified {lm_path} for correct data file path and ensured 'import os', 'import pandas'.")

# --- Reload modules after file modifications ---
# Before re-importing, remove the modules from sys.modules to force a fresh load
if 'omics_data_analysis.sup_func.load_human_metabolites' in sys.modules:
    del sys.modules['omics_data_analysis.sup_func.load_human_metabolites']
if 'omics_data_analysis.Functions.Random_Forest_Analysis' in sys.modules:
    del sys.modules['omics_data_analysis.Functions.Random_Forest_Analysis']

print("Modules removed from sys.modules cache for fresh reload.")

Modified /content/omics_data_analysis/Functions/Random_Forest_Analysis.py for correct imports.
Modified /content/omics_data_analysis/sup_func/load_human_metabolites.py for correct data file path and ensured 'import os', 'import pandas'.
Modules removed from sys.modules cache for fresh reload.


In [10]:
import sys

# Ensure the repository path is in sys.path
if '/content/omics_data_analysis' not in sys.path:
    sys.path.insert(0, '/content/omics_data_analysis')

# Import the specific functions from the freshly loaded modules
from omics_data_analysis.sup_func.Plot_Model_Results import Plot_Model_Results
from omics_data_analysis.sup_func.load_human_metabolites import load_human_metabolites
from omics_data_analysis.Functions.Random_Forest_Analysis import Random_Forest_Analysis

In [13]:
Random_Forest_Analysis(load_human_metabolites, data_frame = True)

In [17]:
analysis = Random_Forest_Analysis(load_human_metabolites, data_frame = True, random_state=42)

In [ ]:
collected, best_vals = analysis.tune_hyperparams(out = True, n_seeds = 10)

In [ ]:
analysis.generate_classifier(parameters = best_vals, out = True)

In [ ]:
analysis.classifier_scores()
analysis.Plot_Results()

In [ ]:
analysis.find_important(use_val = False)
analysis.MI_matrix()

In [ ]:
analysis.partial_dependance_plots(target = "Diabetic_Female", n = 4, pri_pairs = True)
analysis.partial_dependance_plots(target = "Diabetic_Male", n = 4, pri_pairs = True)
analysis.partial_dependance_plots(target = "Control_Female", n = 4, pri_pairs = True)
analysis.partial_dependance_plots(target = "Control_Male", n = 4, pri_pairs = True)